In [ ]:
1. Imports + Setup

In [4]:
# ==============================
# CSL7110 Assignment 2
# Part 1: k-Grams & Jaccard Similarity
# ==============================

import os
import itertools

In [ ]:
2. Load Documents from data/ Folder

In [5]:
# ------------------------------
# Load Documents
# ------------------------------

def load_document(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().strip()

base_path = "data"

documents = {
    "D1": load_document(os.path.join(base_path, "D1.txt")),
    "D2": load_document(os.path.join(base_path, "D2.txt")),
    "D3": load_document(os.path.join(base_path, "D3.txt")),
    "D4": load_document(os.path.join(base_path, "D4.txt")),
}

print("Documents loaded successfully.")
for name, doc in documents.items():
    print(f"{name} length: {len(doc)} characters")

Documents loaded successfully.
D1 length: 1749 characters
D2 length: 1747 characters
D3 length: 2132 characters
D4 length: 1435 characters


In [ ]:
3. k-Gram Generators

In [6]:
# ------------------------------
# k-Gram Generators
# ------------------------------

def char_kgrams(text, k):
    """
    Generate character k-grams (space counts as character).
    Returns a set (duplicates removed).
    """
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def word_kgrams(text, k):
    """
    Generate word k-grams.
    Returns a set of tuples.
    """
    words = text.split()
    return {tuple(words[i:i+k]) for i in range(len(words) - k + 1)}

In [ ]:
4. Construct Required k-Grams

In [7]:
# ------------------------------
# Construct Required k-Grams
# ------------------------------

char_2grams = {name: char_kgrams(doc, 2) for name, doc in documents.items()}
char_3grams = {name: char_kgrams(doc, 3) for name, doc in documents.items()}
word_2grams = {name: word_kgrams(doc, 2) for name, doc in documents.items()}

print("k-grams constructed successfully.")

for name in documents.keys():
    print(f"{name} -> "
          f"Char2: {len(char_2grams[name])}, "
          f"Char3: {len(char_3grams[name])}, "
          f"Word2: {len(word_2grams[name])}")

k-grams constructed successfully.
D1 -> Char2: 263, Char3: 765, Word2: 279
D2 -> Char2: 262, Char3: 762, Word2: 278
D3 -> Char2: 269, Char3: 828, Word2: 337
D4 -> Char2: 255, Char3: 698, Word2: 232


In [ ]:
5. Jaccard Similarity Function

In [8]:
# ------------------------------
# Jaccard Similarity
# ------------------------------

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

In [ ]:
6. Compute All Pairwise Similarities

In [10]:
# ------------------------------
# Pairwise Jaccard Similarities
# ------------------------------

pairs = list(itertools.combinations(documents.keys(), 2))

print("========== Character 2-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(char_2grams[d1], char_2grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

print("\n========== Character 3-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(char_3grams[d1], char_3grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

print("\n========== Word 2-grams ==========")
for d1, d2 in pairs:
    sim = jaccard_similarity(word_2grams[d1], word_2grams[d2])
    print(f"{d1}-{d2}: {sim:.4f}")

========== Character 2-grams ==========
D1-D2: 0.9811
D1-D3: 0.8157
D1-D4: 0.6444
D2-D3: 0.8000
D2-D4: 0.6413
D3-D4: 0.6530

========== Character 3-grams ==========
D1-D2: 0.9780
D1-D3: 0.5804
D1-D4: 0.3051
D2-D3: 0.5680
D2-D4: 0.3059
D3-D4: 0.3121

========== Word 2-grams ==========
D1-D2: 0.9408
D1-D3: 0.1823
D1-D4: 0.0302
D2-D3: 0.1737
D2-D4: 0.0303
D3-D4: 0.0161


In [ ]:
1. Prepare 3-gram Sets for D1 & D2

In [11]:
# ------------------------------
# Part 2: MinHash Setup
# ------------------------------

# Use only character 3-grams
set_D1 = char_3grams["D1"]
set_D2 = char_3grams["D2"]

# Universal set of 3-grams
all_3grams = list(set_D1 | set_D2)

print("Total unique 3-grams:", len(all_3grams))

Total unique 3-grams: 772


In [ ]:
2. Create Hash Family

We use standard universal hash:
h(x)=(a∗x+b)modm

In [12]:
import random

def generate_hash_functions(t, m):
    """
    Generate t hash functions of form:
    h(x) = (a*x + b) % m
    """
    hash_functions = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_functions.append((a, b))
    return hash_functions

In [ ]:
3. Convert 3-grams to Integers

We must hash strings → integers first.

In [13]:
def gram_to_int(gram):
    return abs(hash(gram))

In [ ]:
4. Compute MinHash Signature

In [14]:
def compute_minhash_signature(gram_set, hash_functions, m):
    signature = []
    
    for (a, b) in hash_functions:
        min_hash = float('inf')
        
        for gram in gram_set:
            x = gram_to_int(gram)
            hash_val = (a * x + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
                
        signature.append(min_hash)
        
    return signature

In [ ]:
5. Estimate Jaccard From Signatures

In [15]:
def estimate_jaccard(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])
    return matches / len(sig1)

In [ ]:
6. Run Experiments for Required t Values

In [16]:
# ------------------------------
# MinHash Experiments
# ------------------------------

m = 20000  # must be > 10000

t_values = [20, 60, 150, 300, 600]

results = []

for t in t_values:
    random.seed(42)  # for reproducibility
    
    hash_funcs = generate_hash_functions(t, m)
    
    sig_D1 = compute_minhash_signature(set_D1, hash_funcs, m)
    sig_D2 = compute_minhash_signature(set_D2, hash_funcs, m)
    
    approx_sim = estimate_jaccard(sig_D1, sig_D2)
    
    results.append((t, approx_sim))
    
    print(f"t = {t} → Approx Jaccard = {approx_sim:.4f}")

t = 20 → Approx Jaccard = 0.9000
t = 60 → Approx Jaccard = 0.9500
t = 150 → Approx Jaccard = 0.9667
t = 300 → Approx Jaccard = 0.9767
t = 600 → Approx Jaccard = 0.9683


In [ ]:
7. Compute exact similarity

In [17]:
exact_sim = jaccard_similarity(set_D1, set_D2)
print("Exact Jaccard (3-grams D1-D2):", round(exact_sim, 4))

Exact Jaccard (3-grams D1-D2): 0.978
